# Диагностика подшипников: подготовка сигналов и признаков

Цель работы — загрузить вибрационные сигналы, разбить их на окна и подготовить признаки для моделей регрессии.

Исходное задание: [IImbryk/ml_2026, домашняя работа 3](https://github.com/IImbryk/ml_2026/blob/main/3_homework/3_homework_bearing_diagnostics.ipynb).

Коэффициенты смешивания задают синтетические примеры и не являются измерениями реального износа.

## Подготовка

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.ensemble import RandomForestRegressor

# Поддерживаем запуск из папки ноутбука и из корня репозитория.
data_candidates = [Path("data"), Path("3_homework/data"), Path("3_homework/3_homework/data")]
DATA = next((folder for folder in data_candidates if (folder / "Bearing.mat").is_file()), None)
if DATA is None:
    raise FileNotFoundError("Bearing.mat не найден. Запустите ноутбук из его папки или корня репозитория.")


def check(task, value):
    try:
        if task == 1:
            ok = (isinstance(value, dict) and all(k in value for k in
                  ["normal", "roller", "inner", "outer"])
                  and value["inner"].size == 120_000
                  and round(np.abs(value["inner"]).sum()) == 25057)
        elif task == 2:
            ok = (len(value) == 4 and all(np.asarray(v).shape == (120_000,)
                  for v in value) and round(sum(np.abs(v).sum() for v in value)) == 94201)
        elif task == 3:
            ok = (len(value) == 4 and all(np.asarray(v).shape == (117, 4000)
                  for v in value) and np.array_equal(value[0][0, 1000:], value[0][1, :3000]))
        elif task == 4:
            X, y = value
            ok = (np.asarray(X).shape == (37, 1) and np.asarray(y).shape == (37, 3)
                  and np.allclose(y[0], 0) and np.isclose(y.sum(), 1950)
                  and np.isclose(X[0, 0], 0.0726744486205))
        elif task == 5:
            arr = np.asarray(value, dtype=float)
            ok = arr.shape == (14,) and np.isfinite(arr).all() and round(arr.sum()) == 45545
        elif task == 6:
            arr = np.asarray(value, dtype=float)
            ok = (arr.shape == (37, 1) and np.allclose(arr.mean(axis=0), 0, atol=1e-12)
                  and np.allclose(arr.std(axis=0, ddof=1), 1, atol=1e-12))
        elif task == 7:
            ok = (len(value) == 3 and all(isinstance(v, RandomForestRegressor) for v in value)
                  and [v.n_estimators for v in value] == [400, 400, 600]
                  and [v.max_depth for v in value] == [35, 35, 50])
        else:
            raise ValueError("неизвестный номер")
    except Exception as exc:
        print(f"❌ Задание {task}: {type(exc).__name__}: {exc}")
        return
    print(f"{'✅' if ok else '❌'} Задание {task}" +
          (" выполнено" if ok else ": проверьте результат"))


print("Папка с данными найдена:", DATA.exists())
print("Файлы:", [p.name for p in DATA.iterdir()] if DATA.exists() else [])

## Задание 1. Загрузить MAT-файл

Сохраните словарь из `loadmat(DATA / "Bearing.mat")` в `bearing`.
Ожидаются ключи `normal`, `roller`, `inner`, `outer`.

In [ ]:
bearing = loadmat(DATA / "Bearing.mat")
print("Сигналы в MAT-файле:", [key for key in bearing if not key.startswith("__")])

In [ ]:
check(1, bearing)

## Задание 2. Векторы сигналов

Извлеките четыре сигнала в `normal`, `roller`, `inner`, `outer`.
Метод `.ravel()` превращает матрицу `1 × 120000` в одномерный вектор.

In [ ]:
# ✍ Задание 2
normal = bearing['normal'].ravel()
roller = bearing['roller'].ravel()
inner = bearing['inner'].ravel()
outer = bearing['outer'].ravel()
print("Длины:", [len(v) for v in [normal, roller, inner, outer]] if normal is not Ellipsis else "Заполните код")

In [ ]:
check(2, (normal, roller, inner, outer))

## Задание 3. Окна с перекрытием

Напишите `segment_signal(signal, length=4000, overlap=0.75)`.
Шаг: `int(length * (1 - overlap))`; начала окон: `range(0, len(signal)-length+1, step)`.
Верните массив формы `(число_окон, length)` с помощью `np.stack`.
Получите `segments_normal`, `segments_roller`, `segments_inner`, `segments_outer`.
Какой участок соседних окон общий?

![Перекрытие окон и риск утечки данных](3_homework/visualizations/02_утечка_и_проверка.png)

In [ ]:
def segment_signal(signal, length=4000, overlap=0.75):
    signal = np.asarray(signal)
    if signal.ndim != 1:
        raise ValueError("Ожидается одномерный сигнал")
    if not isinstance(length, (int, np.integer)) or length <= 0:
        raise ValueError("Длина окна должна быть положительным целым числом")
    if not 0 <= overlap < 1:
        raise ValueError("Перекрытие должно лежать в диапазоне [0, 1)")
    step = int(length * (1 - overlap))
    if step < 1 or signal.size < length:
        raise ValueError("Шаг должен быть >= 1, а сигнал — не короче окна")
    starts = range(0, signal.size - length + 1, step)
    return np.stack([signal[start:start + length] for start in starts])

segments_normal, segments_roller, segments_inner, segments_outer = (
    segment_signal(signal) for signal in (normal, roller, inner, outer)
)
print("Форма массива окон:", segments_normal.shape)
print("Шаг: 1000 отсчётов; общая часть соседних окон: 3000 отсчётов (75%)")

In [ ]:
check(3, (segments_normal, segments_roller, segments_inner, segments_outer))

## Задание 4. Synthetic signal mixtures (синтетические смеси сигналов) по одному признаку

Сделайте 37 примеров: один нормальный и по 12 для внутреннего кольца,
внешнего кольца и ролика. Mixing coefficients (коэффициенты смешивания) заданы ниже.
Для каждого примера используйте linear interpolation (линейную интерполяцию):
`normal + mixing_coefficient * (defect - normal)`. Признак — RMS **после вычитания среднего**.
Цель — три числа `[inner, outer, roller]` по учебной шкале 0–100.
Сохраните `X_matrix` формы `(37, 1)` и `y_matrix` формы `(37, 3)`.

![Принцип построения синтетических смесей](3_homework/visualizations/03_synthetic_signal_mixture_and_RMS.png)

In [ ]:
# ✍ Задание 4
mixing_coefficients = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]

def rms(signal):
    return np.sqrt(np.mean((signal - np.mean(signal)) ** 2))

X_values = [rms(normal)]
y_values = [[0.0, 0.0, 0.0]]
for target_col, defect in enumerate([inner, outer, roller]):
    for mixing_coefficient in mixing_coefficients:
        mixture = normal + mixing_coefficient * (defect - normal)
        X_values.append(rms(mixture))
        target = np.zeros(3)
        target[target_col] = mixing_coefficient * 100
        y_values.append(target)

X_matrix = np.array(X_values).reshape(37, 1)
y_matrix = np.array(y_values)
print("X:", getattr(X_matrix, "shape", None), "y:", getattr(y_matrix, "shape", None))

In [ ]:
check(4, (X_matrix, y_matrix))

In [ ]:
# Сравним изменение единственного признака при смешивании сигналов.
fig, ax = plt.subplots(figsize=(8, 4))
for column, label in enumerate(("inner", "outer", "roller")):
    start = 1 + column * len(mixing_coefficients)
    stop = start + len(mixing_coefficients)
    ax.plot(np.array(mixing_coefficients) * 100, X_matrix[start:stop, 0],
            marker="o", markersize=4, label=label)
ax.axhline(X_matrix[0, 0], color="gray", linestyle="--", label="normal")
ax.set(xlabel="Коэффициент смешивания, %", ylabel="RMS после удаления среднего",
       title="Как RMS меняется для разных источников дефекта")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Задание 5. Четырнадцать признаков одного окна

Первые 4000 отсчётов нормального сигнала: RMS, доминирующая частота,
энергии 10 частотных полос по 500 Гц, доля пересечений нуля и отношение
75-го к 25-му процентилю абсолютной амплитуды. Используйте `np.fft.rfft`
и исключите последнюю точку Найквиста, чтобы соответствовать исходному материалу.
Сохраните массив в `features`.

In [ ]:
# ✍ Задание 5
N, fs = 4000, 10_000
x = normal[:N] - np.mean(normal[:N])
rms_value = np.sqrt(np.mean(x ** 2))
amplitude = np.abs(np.fft.rfft(x))[:-1]
frequency = np.fft.rfftfreq(N, d=1/fs)[:-1]
dominant_frequency = frequency[np.argmax(amplitude)]
band_energy = np.array([np.sum(amplitude[(frequency >= i * 500) & (frequency < (i + 1) * 500)] ** 2) for i in range(10)])
zero_crossing_rate = np.sum(np.diff(np.sign(x)) != 0) / (len(x) - 1)
q25, q75 = np.quantile(np.abs(x), [0.25, 0.75])
features = np.array([rms_value, dominant_frequency, *band_energy, zero_crossing_rate, q75 / q25])
print("Число признаков:", len(features) if features is not Ellipsis else "заполните код")

In [ ]:
check(5, features)

## Задание 6. Нормализация признаков

Вычислите среднее и стандартное отклонение **каждого столбца** `X_matrix`.
Если отклонение почти ноль, замените его единицей. Сохраните результат
в `X_scaled`. Используйте `ddof=1`, как в исходном Julia-задании;
самопроверка учитывает эту конвенцию.

In [ ]:
# ✍ Задание 6
means = np.mean(X_matrix, axis=0)
stds = np.std(X_matrix, axis=0, ddof=1)
stds[stds < 1e-10] = 1.0
X_scaled = (X_matrix - means) / stds
print("Среднее:", np.mean(X_scaled, axis=0) if X_scaled is not Ellipsis else "заполните код")

In [ ]:
check(6, X_scaled)

## Задание 7. Создать три случайных леса

Создайте три `RandomForestRegressor`. Для двух колец: 400 деревьев,
максимальная глубина 35. Для ролика: 600 деревьев, глубина 50.
Добавьте `min_samples_split=2`, `min_samples_leaf=1`, а для
воспроизводимости используйте `random_state=42`, `43`, `44`.

Это именно создание моделей: пока `.fit()` не вызван, деревья ещё не обучены.

In [ ]:
# ✍ Задание 7
model_inner = RandomForestRegressor(n_estimators=400, max_depth=35, min_samples_split=2, min_samples_leaf=1, random_state=42)
model_outer = RandomForestRegressor(n_estimators=400, max_depth=35, min_samples_split=2, min_samples_leaf=1, random_state=43)
model_roller = RandomForestRegressor(n_estimators=600, max_depth=50, min_samples_split=2, min_samples_leaf=1, random_state=44)
print("Деревьев:", [m.n_estimators for m in (model_inner, model_outer, model_roller)]
      if model_inner is not Ellipsis else "заполните код")

In [ ]:
check(7, (model_inner, model_outer, model_roller))

## Выводы и ограничения

- Из каждого сигнала получено 117 окон длиной 4000 с шагом 1000. Соседние окна имеют 3000 общих отсчётов, поэтому случайное разделение таких окон может привести к утечке. Для оценки качества нужны независимые записи либо непересекающиеся временные участки, выделенные до сегментации.
- Матрица из 37 синтетических примеров содержит один признак — RMS после удаления среднего. Он характеризует амплитуду колебаний, но сам по себе не гарантирует различение типов дефекта.
- Вектор из 14 признаков рассчитан отдельно для одного нормального окна и пока не используется в `X_matrix`.
- Три случайных леса созданы с заданными параметрами, но не обучены. Оценки качества и выводов о диагностике реального оборудования в этой работе нет.
- Шкала 0–100 описывает коэффициент смешивания сигналов, а не физический процент износа.